# Overpass API (OpenStreetMap)

**Endpoint:** `http://overpass-api.de/api/interpreter` (port 80)  
**Method:** POST with Overpass QL query

**Purpose:** Count hospitals, airports and police stations within 150 km of the epicenter.  
Uses `nwr` (node + way + relation) to capture all OSM object types.

**Used in:** `services/flink-enrichment/enrichment_job.py` → `fetch_overpass()`

In [6]:
import http.client
import urllib.parse
import json

# Example coordinates: Athens, Greece
LAT    = 37.98
LON    = 23.73
RADIUS = 300000  # 150 km in metres

HOST     = "overpass-api.de"
ENDPOINT = "/api/interpreter"


# def overpass_post(query: str) -> dict:
#     body = urllib.parse.urlencode({"data": query}).encode("utf-8")
#     headers = {
#         "Content-Type": "application/x-www-form-urlencoded",
#         "User-Agent":   "EarthquakeMonitor/1.0",
#     }
#     conn = http.client.HTTPConnection(HOST, 80, timeout=60)
#     conn.request("POST", ENDPOINT, body=body, headers=headers)
#     resp = conn.getresponse()
#     if resp.status != 200:
#         raise RuntimeError(f"Overpass API returned HTTP {resp.status}: {resp.reason}")
#     return json.loads(resp.read().decode("utf-8"))


# print(f"Host     : {HOST}")
# print(f"Endpoint : {ENDPOINT}")

In [12]:
def fetch_overpass(lat: float, lon: float, radius: float) -> dict:
    try:
        query = (
            "[out:json][timeout:60];\n"
            "(\n"
            f'  nwr["amenity"="hospital"](around:{radius},{lat},{lon});\n'
            f'  nwr["aeroway"="aerodrome"](around:{radius},{lat},{lon});\n'
            f'  nwr["amenity"="police"](around:{radius},{lat},{lon});\n'
            ");\n"
            "out tags;"
        )
        body = urllib.parse.urlencode({"data": query}).encode("utf-8")
        conn = http.client.HTTPConnection(HOST, 80, timeout=60)
        conn.request("POST", ENDPOINT, body=body, headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "User-Agent":   "EarthquakeMonitor/1.0",
        })
        resp = conn.getresponse()
        if resp.status != 200:
            raise RuntimeError(f"Overpass API returned HTTP {resp.status}: {resp.reason}")
        return json.loads(resp.read().decode("utf-8")).get("elements", [])
    except Exception as e:
        print(f"Overpass fetch failed: {e}", flush=True)
        return {}

In [14]:
overpass = fetch_overpass(LAT, LON, RADIUS)
# print(overpass)

In [15]:
# hospitals       = [e for e in overpass.get("elements", []) if e.get("tags", {}).get("amenity") == "hospital"]
# airports        = [e for e in overpass.get("elements", []) if e.get("tags", {}).get("aeroway") == "aerodrome"]
# police_stations = [e for e in overpass.get("elements", []) if e.get("tags", {}).get("amenity") == "police"]

# infrastructure_counts = {
#     "hospitals_count":       len(hospitals),
#     "airports_count":        len(airports),
#     "police_stations_count": len(police_stations),
# }

# infrastructure = {
#     "hospitals":       hospitals,
#     "airports":        airports,
#     "police_stations": police_stations,
# }

def _json_default(obj):
    if isinstance(obj, bytes):
        return obj.decode("utf-8")
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

# print(json.dumps(infrastructure_counts, indent=2, ensure_ascii=False))
print(json.dumps(overpass, indent=2, ensure_ascii=False))
# if hospitals:
#     print(json.dumps(hospitals[0], indent=2, ensure_ascii=False, default=_json_default))

[
  {
    "type": "node",
    "id": 279415483,
    "tags": {
      "amenity": "police",
      "int_name": "Tourist Police",
      "name": "Τουριστική Αστυνομια",
      "name:el": "Τουριστική Αστυνομια"
    }
  },
  {
    "type": "node",
    "id": 347294937,
    "tags": {
      "amenity": "police",
      "name": "Αστυνομικό Τμήμα"
    }
  },
  {
    "type": "node",
    "id": 428151392,
    "tags": {
      "aeroway": "aerodrome",
      "icao": "LGMG",
      "name": "Αεροδρόμιο Μεγαρέων",
      "name:el": "Αεροδρόμιο Μεγαρέων",
      "name:en": "Megara Air Base & Airport",
      "note": "mixed use: military in origin, but also serves as the G/A terminal for Athens",
      "wikidata": "Q12693913",
      "wikipedia": "el:Μονάδα Εξυπηρέτησης Αεροσκαφών Γενικής Αεροπορίας"
    }
  },
  {
    "type": "node",
    "id": 438011333,
    "tags": {
      "amenity": "police"
    }
  },
  {
    "type": "node",
    "id": 458252633,
    "tags": {
      "amenity": "police"
    }
  },
  {
    "type": "nod